# Data Science for Business - Multilayer Perceptron (MLP) on Ames Housing with Scikit-learn

*Session 8 · ISLP 10.1–10.2 · 10.7 Fitting a Neural Network · Lab 10.9.1*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.linear_model import LassoCV
from sklearn.neural_network import MLPRegressor

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load training data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_08/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

First, we will remove some columns that are not useful for our task.

In [ ]:
data = data.drop(['YrSold', 'MoSold', 'SaleCondition', 'SaleType'], axis=1)

Next, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets. We use the same split as in Session 5, so that we can compare the neural network with the lasso.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Finally, we will do some feature engineering. It is important to use only information from the training set for feature engineering, and then mechanistically repeat these steps on the test set.

Typically, feature engineering depends strongly on the datatype of the variables. Hence, we will first determine which variables are categorical and which are numerical. Subsequently, we will transform these variables separately.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

The categorical variables must be transformed into numerical representations, e.g., by one-hot encoding them.

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc.fit(X_train[categorical_features])

X_train_cat = enc.transform(X_train[categorical_features])
X_test_cat = enc.transform(X_test[categorical_features])

X_train_cat = pd.DataFrame(X_train_cat, columns=enc.get_feature_names_out(categorical_features))
X_test_cat = pd.DataFrame(X_test_cat, columns=enc.get_feature_names_out(categorical_features))

The numerical variables will be standardized, that is, we will subtract the mean and divide by the standard deviation. Neural networks are trained by gradient descent, which works much better when all inputs are on a similar scale (ISLP 10.9.1 standardizes the predictors as well).

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[numerical_features])

X_train_num = scaler.transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

X_train_num = pd.DataFrame(X_train_num, columns=numerical_features)
X_test_num = pd.DataFrame(X_test_num, columns=numerical_features)

Let's fuse the engineered categorical and numerical variables again.

In [ ]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

In [ ]:
X_train.shape

The same holds for the response: sale prices are in the hundreds of thousands and right-skewed, so a squared-error loss on the raw prices produces huge, poorly scaled gradients. We therefore train on the log of the price and transform the predictions back with `np.exp` before computing the RMSE in dollars.

In [ ]:
y_train_log = np.log(y_train)

## Baseline: lasso

Like ISLP Lab 10.9.1, we first fit a linear model as a baseline. We use the lasso from Session 5 (ISLP 6.2.2), with λ chosen by 5-fold cross-validation, on the same predictors but now with the log price as response, like the network.

In [ ]:
model_lasso = LassoCV(cv=5, random_state=42)
model_lasso.fit(X_train, y_train_log)

In [ ]:
pred_test_lasso = np.exp(model_lasso.predict(X_test))
rmse_test_lasso = root_mean_squared_error(y_test, pred_test_lasso)
print('Lasso R2 on test set:', round(r2_score(y_test, pred_test_lasso), 2))
print('Lasso RMSE on test set:', round(rmse_test_lasso, 2))

## Neural network

In the following, we will create a neural network with one hidden layer comprising 256 hidden units. Each hidden unit computes $A_k = g(w_{k0} + \sum_j w_{kj}X_j)$ with the ReLU activation function $g$; the output is a linear combination of the $A_k$ (ISLP 10.1, equations 10.1 and 10.2).

The network is trained with Adam, a popular variant of stochastic gradient descent (ISLP 10.7.2), with learning rate ρ = 0.001 (`learning_rate_init`). One epoch is one pass over the training data; we allow at most 1000 epochs. With `early_stopping=True`, scikit-learn sets aside 10% of the training data as a validation set and stops when the validation score has not improved for 10 epochs (early stopping, ISLP 10.7.2). Note that `alpha` in `MLPRegressor` is the strength of an L2 (ridge) penalty on the weights, yet another meaning of "alpha".

In [ ]:
model_mlp = MLPRegressor(hidden_layer_sizes=(256,), activation='relu', solver='adam', learning_rate_init=0.001, alpha=0.0001,
                         max_iter=1000, early_stopping=True, validation_fraction=0.1, random_state=42)

In [ ]:
model_mlp.fit(X_train, y_train_log)

Let's look at the training process: the training loss (squared error on the log scale) and the R² on the internal validation set per epoch.

In [ ]:
print('Number of epochs:', model_mlp.n_iter_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(model_mlp.loss_curve_)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss')
axes[1].plot(model_mlp.validation_scores_)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation R2')
plt.tight_layout()
plt.show()

Now we evaluate the network on the training and test set. Remember to transform the predictions back to dollars.

In [ ]:
# Training data
pred_train = np.exp(model_mlp.predict(X_train))
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = np.exp(model_mlp.predict(X_test))
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

How to read these numbers: a big gap between training and test performance indicates overfitting; similar but poor values indicate underfitting or an unfinished optimisation.

Finally, let's compare the network with the lasso baseline.

In [ ]:
pd.DataFrame({'Model': ['Lasso', 'MLP'], 'Test RMSE': [rmse_test_lasso, rmse_test]}).round(0)

Here, the network does not beat the lasso. ISLP 10.6 ("When to Use Deep Learning") makes a similar point: neural networks shine when the sample size is large and the signal is complex (images, text, ...), while for small tabular datasets like ours (about 2,300 training observations), simpler models often perform as well or better and are easier to interpret.

## Your turn!

**Your task:** Try to beat the lasso. Vary the architecture (`hidden_layer_sizes`, e.g., `(256, 128)` for two hidden layers), the L2 penalty `alpha` (e.g., 0.01, 0.1, 1) and the learning rate `learning_rate_init`. Keep `early_stopping=True` and compare the test RMSE with the lasso baseline.

In [ ]:
# YOUR CODE HERE